# Test de la stratégie A : LLM seul

## Objectif
Évaluer la stratégie A, reposant exclusivement sur l’utilisation d’un modèle de langage (LLM) pour répondre aux questions d’une FAQ administrative, sans recours à une base documentaire externe.  
Cette approche vise à mesurer les capacités intrinsèques du LLM à produire des réponses pertinentes, cohérentes et exploitables dans un contexte de service public.

Les tests sont réalisés à l’aide des modèles **Qwen/Qwen2.5-7B-Instruct** et **mistralai/Mistral-7B-Instruct-v0.2**.

## Critères de vérification
- **Exactitude** des réponses fournies par le LLM.
- **Minimisation des hallucinations**, notamment sur les procédures locales.
- **Pertinence** des réponses par rapport aux questions posées.
- **Latence** du modèle lors de la génération des réponses.
- **Qualité de la formulation** et du **ton institutionnel** des réponses.


In [6]:
from pathlib import Path

# chargement des FAQ
root_path= Path().resolve().parents[0]
faq_path = root_path/"FAQ_Base.json"
print(root_path, faq_path)

D:\ProjectFolderDevAI_2025-2026\Projet_FAQ_Intelligent\Assistant_FAQ_Intelligent D:\ProjectFolderDevAI_2025-2026\Projet_FAQ_Intelligent\Assistant_FAQ_Intelligent\FAQ_Base.json


In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
import os


load_dotenv()
# Initialisation du client avec le token d'authentification
token_benchmark_faq = os.getenv("token_benchmark_faq")

login(token_benchmark_faq)


In [4]:
from huggingface_hub import InferenceClient

print(f"Using token: {token_benchmark_faq[:5]}...")  # Affiche les 5 premiers caractères du token pour vérification

client = InferenceClient(
                         token=token_benchmark_faq,
                       
                        )

# Bonne méthode : chat_completion (pour Mistral, Zephyr, etc.)
messages = [
    {"role": "system", "content": (
        
        """
            Tu es un assistant virtuel expert pour la Communauté de Communes Val de Loire Numérique.
            Ton rôle est de répondre EXCLUSIVEMENT aux questions concernant la collectivité territoriale et les démarches administratives.

            Règles de politesse OBLIGATOIRES :
            1. Commence toujours par "Bonjour,"
            2. Termine toujours par une formule de politesse (ex : "Veuillez reformuler votre demande").

            Règle en cas de hors sujet :
            Si la question ne concerne pas la collectivité, tu dois répondre poliment mais fermement :
            "Bonjour, je ne suis pas habilité à répondre à ce genre de question. Veuillez reformuler votre demande."

            Ne donne aucune autre explication si le sujet est hors périmètre.
        """
)},
    {"role": "user", "content": "Bonjour, Que faire pour déclarer une naissance ?"}
]

response = client.chat_completion(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=messages,
    max_tokens=250
)

print(response.choices[0].message.content)

Using token: hf_AD...
Bonjour,

Pour déclarer une naissance, vous devez vous rendre au service de l'état civil de la commune où le bébé a été né. Vous aurez besoin de présenter certains documents tels que l'acte de naissance émis par l'hôpital, les pièces d'identité des parents, et éventuellement le livret de famille.

Si vous avez des questions supplémentaires ou besoin d'aide pour les démarches, n'hésitez pas à me le faire savoir.

Veuillez reformuler votre demande si vous avez besoin d'informations sur d'autres démarches administratives.


In [19]:
from huggingface_hub import InferenceClient

print(f"Using token: {token_benchmark_faq[:5]}...")  # Affiche les 5 premiers caractères du token pour vérification

client = InferenceClient(
                        token=token_benchmark_faq,
                        provider="featherless-ai"
                        )

SYSTEM_PROMPT = (
    "Tu es un assistant virtuel officiel de la Communauté de Communes Val de Loire Numérique. "
    "Tu réponds EXCLUSIVEMENT aux questions liées aux démarches administratives de la collectivité "
    "(état civil, urbanisme, déchets, transports, services aux habitants).\n\n"

    "RÈGLES DE SORTIE (OBLIGATOIRES)\n"
    "1) Réponds toujours en français.\n"
    "2) Commence toujours par : « Bonjour, »\n"
    "3) Réponse courte et utile : 3 à 8 phrases maximum.\n"
    "4) Si la question est hors périmètre, réponds UNIQUEMENT par la phrase suivante, mot pour mot, "
    "sans ajouter d'explication ni d'excuse :\n"
    "« Bonjour, je ne suis pas habilité à répondre à ce genre de question. Veuillez reformuler votre demande. »\n"
)


# Bonne méthode : chat_completion (pour Mistral, Zephyr, etc.)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Bonjour, quelle est le role d'une tarte ?"}
]

response = client.chat_completion(
    model="mistralai/Mistral-7B-Instruct-v0.2",
    messages=messages,
    max_tokens=250
)

print(response.choices[0].message.content)

Using token: hf_AD...
 Bonjour, je ne suis pas habilité à répondre à cette question. Vous pouvez rechercher de plus amples informations sur la signification de la terme "tarte" dans le contexte culinaire ou other. If you have a question related to the administrative procedures of our community, please ask.


In [30]:
from huggingface_hub import InferenceClient

print(f"Using token: {token_benchmark_faq[:5]}...")  # Affiche les 5 premiers caractères du token pour vérification

client = InferenceClient(
                         token=token_benchmark_faq                      
                        )

SYSTEM_PROMPT = (
    "Tu es un assistant virtuel officiel de la Communauté de Communes Val de Loire Numérique. "
    "Tu réponds EXCLUSIVEMENT aux questions liées aux démarches administratives de la collectivité "
    "(état civil, urbanisme, déchets, transports, services aux habitants).\n\n"

    "PRINCIPES FONDAMENTAUX\n"
    "- Tu privilégies toujours l'exactitude à l'exhaustivité.\n"
    "- En cas de doute, tu demandes une précision plutôt que de faire une supposition.\n"
    "- Tu n'inventes JAMAIS de canal de démarche (en ligne, téléphone, guichet, sur place), "
    "ni de procédure, ni de document requis.\n\n"

    "RÈGLES DE SORTIE (OBLIGATOIRES)\n"
    "1) Réponds toujours en français.\n"
    "2) Commence toujours par : « Bonjour, »\n"
    "3) Structure obligatoirement ta réponse avec les sections suivantes :\n"
    "   - Démarche générale\n"
    "   - Éléments dépendant de la commune\n"
    "   - Précisions nécessaires\n"
    "4) Dans « Démarche générale », décris uniquement l'objectif administratif, sans citer de moyen, canal ou outil.\n"
    "5) Dans « Éléments dépendant de la commune », indique que les modalités varient, sans donner d'exemple.\n"
    "6) Dans « Précisions nécessaires », pose uniquement des questions factuelles.\n"
    "7) Si la question concerne une démarche administrative concrète, tu n'as PAS le droit de décrire la démarche.Tu dois uniquement demander des précisions.\n"
    "8) Si la question est hors périmètre, réponds UNIQUEMENT par la phrase suivante, mot pour mot :\n"
    "« Bonjour, je ne suis pas habilité à répondre à ce genre de question. Veuillez reformuler votre demande. »\n"
)



# Bonne méthode : chat_completion (pour Mistral, Zephyr, etc.)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Bonjour, comment s'inscrire pour accéder à la déchetterie ?"}]

response = client.chat_completion(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=messages,
    temperature=0.1,
    max_tokens=200
)

print(response.choices[0].message.content)

Using token: hf_AD...
Bonjour,

Démarche générale : L'inscription pour accéder à la déchetterie vise à identifier les usagers et assurer le bon fonctionnement des installations.

Éléments dépendant de la commune : Les modalités d'inscription peuvent varier selon la commune. Il est recommandé de vérifier auprès de la mairie ou de la déchetterie.

Précisions nécessaires : Pourriez-vous me préciser la commune concernée ?


In [34]:
from huggingface_hub import InferenceClient

print(f"Using token: {token_benchmark_faq[:5]}...")  # Affiche les 5 premiers caractères du token pour vérification

client = InferenceClient(
                         token=token_benchmark_faq,
                         provider="featherless-ai"                     
                        )

SYSTEM_PROMPT = (
    "Bonjour, je suis désolé, mais je ne peux pas répondre à cette question car elle n'est pas liée aux services de la collectivité Val de Loire."
                )



# Bonne méthode : chat_completion (pour Mistral, Zephyr, etc.)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Bonjour, comment s'inscrire pour accéder à la déchetterie ?"}]

response = client.chat_completion(
    model="mistralai/Mistral-7B-Instruct-v0.2",
    messages=messages,
    temperature=0.1,
    max_tokens=200
)

print(response.choices[0].message.content)

Using token: hf_AD...
 Je suis désolé, mais je ne peux pas répondre à cette question car je ne dispose pas des informations nécessaires pour vous inscrire à la déchetterie de la collectivité Val de Loire. Il est possible que vous deviez contacter directement le service des déchets de cette collectivité pour obtenir les informations pratiques et les modalités d'inscription. Vous pouvez consulter leur site web ou appeler leur numéro de contact pour plus d'information.


In [35]:
from huggingface_hub import InferenceClient

print(f"Using token: {token_benchmark_faq[:5]}...")  # Affiche les 5 premiers caractères du token pour vérification

client = InferenceClient(
                         token=token_benchmark_faq,
                         provider="featherless-ai"                     
                        )

SYSTEM_PROMPT = (
    "Tu es un assistant virtuel officiel de la Communauté de Communes Val de Loire Numérique.\n"
    "Tu réponds uniquement aux questions liées aux démarches administratives de la collectivité.\n"
    "Si une question est hors périmètre, tu réponds uniquement par la phrase suivante, mot pour mot :\n"
    "« Bonjour, je ne suis pas habilité à répondre à cette question. Veuillez reformuler votre demande. »"
)




# Bonne méthode : chat_completion (pour Mistral, Zephyr, etc.)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Bonjour, comment s'inscrire pour accéder à la déchetterie ?"}]

response = client.chat_completion(
    model="mistralai/Mistral-7B-Instruct-v0.2",
    messages=messages,
    temperature=0.1,
    max_tokens=200
)

print(response.choices[0].message.content)

Using token: hf_AD...
 Bonjour, pour s'inscrire et accéder à la déchetterie de la Communauté de Communes Val de Loire Numérique, vous devez suivre les étapes suivantes :

1. Consultez notre site web ou contactez notre service client pour obtenir les informations les plus récentes sur les documents à fournir et les procedures d'inscription.
2. Renseignez les documents exigés, tels que votre preuve d'identité et d'adresse, et une preuve de domicile si cela est requis.
3. Envoyez ou déposez les documents à notre service administratif ou à notre mairie.
4. Nous traiterons votre demande et vous notifierons par la poste ou par courrier électronique lorsque votre inscription est terminée
